In [ ]:
#Part (a): Preprocessing the Dataset

In [74]:
import pandas as pd

# Load the dataset
file_path = '/Users/shiveshrajsahu/Desktop/Cs767/cs767A1_SAHU_SHIVESH_HW10/hin.csv'
dataset = pd.read_csv(file_path)

# Display the first few lines of the dataset to understand its structure
dataset.head()


,Wow!,वाह!,CC-BY 2.0 (France) Attribution: tatoeba.org #52027 (Zifre) & #6179147 (fastrizwaan)
0,Duck!,झुको!,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Duck!,बतख़!,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Help!,बचाओ!,CC-BY 2.0 (France) Attribution: tatoeba.org #4...
3,Jump.,उछलो.,CC-BY 2.0 (France) Attribution: tatoeba.org #6...
4,Jump.,कूदो.,CC-BY 2.0 (France) Attribution: tatoeba.org #6...


In [75]:
# Load and examine the first few lines of the dataset to understand its structure
#file_path = '/Users/shiveshrajsahu/Desktop/Cs767/cs767A1_SAHU_SHIVESH_HW10/hin.csv'

# Reading the first few lines of the file
with open(file_path, 'r', encoding='utf-8') as file:
    first_lines = [next(file) for _ in range(5)]

first_lines



['Wow!,वाह!,CC-BY 2.0 (France) Attribution: tatoeba.org #52027 (Zifre) & #6179147 (fastrizwaan)\n',
 'Duck!,झुको!,CC-BY 2.0 (France) Attribution: tatoeba.org #280158 (CM) & #6179041 (fastrizwaan)\n',
 'Duck!,बतख़!,CC-BY 2.0 (France) Attribution: tatoeba.org #280158 (CM) & #6179042 (fastrizwaan)\n',
 'Help!,बचाओ!,CC-BY 2.0 (France) Attribution: tatoeba.org #435084 (lukaszpp) & #459377 (minshirui)\n',
 'Jump.,उछलो.,CC-BY 2.0 (France) Attribution: tatoeba.org #631038 (Shishir) & #6179121 (fastrizwaan)\n']

In [76]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import random

# Load the dataset
df = pd.read_csv(file_path, header=None, usecols=[0, 1], names=['English', 'Hindi'])

# Data cleaning function
def clean_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence)  # creating a space between words and punctuation
    sentence = re.sub(r'[" "]+', " ", sentence)         # replacing multiple spaces with a single space
    sentence = re.sub(r"[^a-zA-Z?.!,¿]+", " ", sentence)  # removing all characters except a-z and punctuation
    return sentence.strip()

# Applying cleaning function to both English and Hindi sentences
df['English'] = df['English'].apply(clean_sentence)
df['Hindi'] = df['Hindi'].apply(lambda x: x.strip().lower())

# Splitting the dataset
train, test = train_test_split(df, test_size=0.15, random_state=42)
train, val = train_test_split(train, test_size=45/85, random_state=42)  # 45% of 85% is approximately 53% of the original

# Showing the sizes of each dataset and a few samples
train_size = len(train)
val_size = len(val)
test_size = len(test)

train.head(), val.head(), test.head(), (train_size, val_size, test_size)



(                                      English  \
 1372            she was robbed of her purse .   
 2217   i want to know why tom is doing this .   
 792                  i seem to have a fever .   
 2489  he is always here between and o clock .   
 894                 i really like city life .   
 
                                              Hindi  
 1372                 उसका पर्स उससे चुरा लिया गया।  
 2217     मुझे यह बताओ, कि टॉम ऐसा कर क्यों रहा है।  
 792                       मुझे बुखार जैसा लगता है।  
 2489  वह हमेशा यहाँ पाँच से छः बजे के बीच होता है।  
 894                  मुझे शहर की ज़िन्दगी पसंद है।  ,
                                         English  \
 293                          it s april first .   
 2427  we should try to understand one another .   
 759                     you reap what you sow .   
 86                                he stood up .   
 2353   she is living in some village in india .   
 
                                            Hindi  
 293   

In [77]:
# Save the split data to files
train.to_csv('train_set.csv', index=False)
validation.to_csv('validation_set.csv', index=False)
test.to_csv('test_set.csv', index=False)

In [78]:
#Part (b): Building and Training the Model

In [79]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Attention


In [80]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Embedding, Concatenate, Attention

In [81]:
# Parameters
max_vocab_size = 2000
max_length = 70
embedding_dim = 256
lstm_units = 512

In [82]:
# Tokenization and padding function
def tokenize_and_pad(texts, max_vocab_size, max_length):
    tokenizer = Tokenizer(num_words=max_vocab_size, oov_token="<OOV>")
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(sequences, maxlen=max_length, padding='post')
    return tokenizer, padded

In [83]:
# Tokenize and pad the sentences for each dataset
eng_tokenizer, train_eng = tokenize_and_pad(train['English'], max_vocab_size, max_length)
hin_tokenizer, train_hin = tokenize_and_pad(train['Hindi'], max_vocab_size, max_length)

_, val_eng = tokenize_and_pad(val['English'], max_vocab_size, max_length)
_, val_hin = tokenize_and_pad(val['Hindi'], max_vocab_size, max_length)

_, test_eng = tokenize_and_pad(test['English'], max_vocab_size, max_length)
_, test_hin = tokenize_and_pad(test['Hindi'], max_vocab_size, max_length)

In [84]:
# Building the encoder-decoder model with attention
encoder_inputs = Input(shape=(max_length,))
encoder_embedding = Embedding(input_dim=max_vocab_size, output_dim=embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(lstm_units, return_state=True, return_sequences=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(max_length,))
decoder_embedding = Embedding(input_dim=max_vocab_size, output_dim=embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

attention = Attention()
attention_out = attention([decoder_outputs, encoder_outputs])

decoder_concat = Concatenate(axis=-1)([decoder_outputs, attention_out])
decoder_dense = Dense(max_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_concat)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')


#test run with epochs = 2 
# Train the model
model.fit([train_eng, train_hin], train_hin, epochs=2, batch_size=64, validation_data=([val_eng, val_hin], val_hin))

# Save the model (optional)
model.save('translation_model.h5')

Epoch 1/2


W0000 00:00:1700206051.281331       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


19/19 [==============================] - ETA: 0s - loss: 1.4114

W0000 00:00:1700206060.748085       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


19/19 [==============================] - 14s 718ms/step - loss: 1.4114 - val_loss: 0.6962
Epoch 2/2
19/19 [==============================] - 13s 716ms/step - loss: 0.6626 - val_loss: 0.7504


/Users/shiveshrajsahu/anaconda3/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [99]:
# now epochs=20
# Train the model
model.fit([train_eng, train_hin], train_hin, epochs=20, batch_size=64, validation_data=([val_eng, val_hin], val_hin))

# Save the model (optional)
model.save('translation_model.h5')

Epoch 1/20


W0000 00:00:1700209285.376251       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


19/19 [==============================] - ETA: 0s - loss: 0.6222

W0000 00:00:1700209294.560141       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


19/19 [==============================] - 14s 694ms/step - loss: 0.6222 - val_loss: 0.6723
Epoch 2/20
19/19 [==============================] - 13s 685ms/step - loss: 0.5865 - val_loss: 0.6606
Epoch 3/20
19/19 [==============================] - 13s 681ms/step - loss: 0.5572 - val_loss: 0.6450
Epoch 4/20
19/19 [==============================] - 13s 681ms/step - loss: 0.5199 - val_loss: 0.6172
Epoch 5/20
19/19 [==============================] - 13s 694ms/step - loss: 0.4843 - val_loss: 0.5895
Epoch 6/20
19/19 [==============================] - 13s 692ms/step - loss: 0.4482 - val_loss: 0.5732
Epoch 7/20
19/19 [==============================] - 13s 703ms/step - loss: 0.4118 - val_loss: 0.5552
Epoch 8/20
19/19 [==============================] - 13s 713ms/step - loss: 0.3751 - val_loss: 0.5291
Epoch 9/20
19/19 [==============================] - 13s 718ms/step - loss: 0.3407 - val_loss: 0.5033
Epoch 10/20
19/19 [==============================] - 13s 697ms/step - loss: 0.3118 - val_loss: 0.4791


In [100]:
model.summary()


Model: "model_7"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_11 (InputLayer)       [(None, 70)]                 0         []                            
                                                                                                  
 input_12 (InputLayer)       [(None, 70)]                 0         []                            
                                                                                                  
 embedding_6 (Embedding)     (None, 70, 256)              512000    ['input_11[0][0]']            
                                                                                                  
 embedding_7 (Embedding)     (None, 70, 256)              512000    ['input_12[0][0]']            
                                                                                            

In [101]:
# Part c: Evaluation

In [102]:
import numpy as np
from tensorflow.keras.models import load_model
from nltk.translate.bleu_score import sentence_bleu
import nltk


In [103]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/shiveshrajsahu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [104]:
# Load your trained model
model = load_model('translation_model.h5')

In [105]:
#Function to convert a sequence of integers back to a sentence
def seq_to_sentence(sequence, tokenizer):
    words = [tokenizer.index_word[i] for i in sequence if i != 0]
    return ' '.join(words)

In [106]:
# Function to generate a translation from a model
def translate(model, source, eng_tokenizer, hin_tokenizer):
    # Convert the source text to a sequence
    source_seq = eng_tokenizer.texts_to_sequences([source])
    source_seq = pad_sequences(source_seq, maxlen=max_length, padding='post')
    
    # Predict the sequence
    prediction = model.predict(source_seq, verbose=0)[0]
    predicted_seq = np.argmax(prediction, axis=1)
    
    # Convert the sequence of integers to a sentence
    translation = seq_to_sentence(predicted_seq, hin_tokenizer)
    return translation


In [107]:
test_eng 

array([[ 12,  15, 181, ...,   0,   0,   0],
       [ 17,  15, 130, ...,   0,   0,   0],
       [323, 324,  43, ...,   0,   0,   0],
       ...,
       [ 20,   5,  13, ...,   0,   0,   0],
       [  2,  23, 867, ...,   0,   0,   0],
       [ 30,  25, 115, ...,   0,   0,   0]], dtype=int32)

In [108]:
test_hin

array([[  65,  363,   37, ...,    0,    0,    0],
       [   4,   39,  209, ...,    0,    0,    0],
       [   4,  365,   11, ...,    0,    0,    0],
       ...,
       [  13,  115,  299, ...,    0,    0,    0],
       [   3, 1030,    5, ...,    0,    0,    0],
       [1032,    5,   15, ...,    0,    0,    0]], dtype=int32)

In [109]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Modify the translate function
def translate(model, source, eng_tokenizer, hin_tokenizer, max_length):
    # Convert the source text to a sequence
    source_seq = eng_tokenizer.texts_to_sequences([source])
    source_seq = pad_sequences(source_seq, maxlen=max_length, padding='post')

    # Create a placeholder for the decoder input
    # This is a hack and may not work for all models
    decoder_input_seq = np.zeros((1, max_length))

    # Predict the sequence
    prediction = model.predict([source_seq, decoder_input_seq], verbose=0)[0]
    predicted_seq = np.argmax(prediction, axis=1)
    
    # Convert the sequence of integers to a sentence
    translation = seq_to_sentence(predicted_seq, hin_tokenizer)
    return translation

# Your evaluation loop should remain the same, but include max_length in the translate call
max_length = 70  # or whatever the max length of your sequences is


In [64]:
# Function to map a tokenized sequence back to a sentence
def tokens_to_text(tokens, tokenizer):
    return ' '.join([tokenizer.index_word.get(token, '') for token in tokens if token != 0])

# Adjust the evaluation loop
bleu_scores = []
for i in range(len(test_eng)):
    # Convert tokenized sequences back to text
    source_text = tokens_to_text(test_eng[i], eng_tokenizer)
    actual_text = tokens_to_text(test_hin[i], hin_tokenizer)
    
    # Translate the text
    predicted = translate(model, source_text, eng_tokenizer, hin_tokenizer)
    
    # Calculate BLEU score
    actual = [actual_text.split()]
    predicted = predicted.split()
    score = sentence_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))
    bleu_scores.append(score)
    
    print(f'Source: {source_text}')
    print(f'Actual: {actual}')
    print(f'Predicted: {predicted}')
    print(f'BLEU Score: {score}\n')


TypeError: translate() missing 1 required positional argument: 'max_length'

In [95]:
#okay let me try this again 

In [110]:
# Define max_length
max_length = 70  # This should be the same max_length used in model training

# Adjust the evaluation loop to include max_length in the translate call
bleu_scores = []
for i in range(len(test_eng)):
    # Convert tokenized sequences back to text
    source_text = tokens_to_text(test_eng[i], eng_tokenizer)
    actual_text = tokens_to_text(test_hin[i], hin_tokenizer)
    
    # Translate the text
    predicted = translate(model, source_text, eng_tokenizer, hin_tokenizer, max_length)
    
    # Calculate BLEU score
    actual = [actual_text.split()]
    predicted = predicted.split()
    score = sentence_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))
    bleu_scores.append(score)
    
    print(f'Source: {source_text}')
    print(f'Actual: {actual}')
    print(f'Predicted: {predicted}')
    print(f'BLEU Score: {score}\n')


W0000 00:00:1700209576.428839       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


Source: in have way saw to two knows on very
Actual: [['तो', 'ज़रूरत', 'अपनी', 'है।', 'हूँ।', 'ने', 'मुझसे', 'था']]
Predicted: []
BLEU Score: 0

Source: s have book lose you lost we she believe
Actual: [['में', 'उसे', 'नौकरी', 'है', 'यह', 'की', 'उनके', 'है।']]
Predicted: []
BLEU Score: 0

Source: better robbed and see as back
Actual: [['में', 'कुत्ता', 'के', 'अच्छे', 'मुझे', 'गाड़ी', 'वह', 'उससे', 'तैरना']]
Predicted: []
BLEU Score: 0

Source: to been fever you the written
Actual: [['हूँ।', 'भूख', 'पर', 'विश्वास', 'अपने', 'था']]
Predicted: []
BLEU Score: 0

Source: i letter his at is been was speak is has it was in
Actual: [['इस', 'था।', 'को', 'बच्चों', 'आप', 'हैं।', 'अपने', 'माँ', 'का', 'लिए', 'कैसे', 'वह', 'देर']]
Predicted: []
BLEU Score: 0

Source: i my talk of up you watch let some the baby
Actual: [['से', 'बजे', 'स्कूल', 'की', 'छोटे', 'मेरे', 'एक', 'है', 'अब', 'अपने', 'है।']]
Predicted: []
BLEU Score: 0

Source: mary just that when
Actual: [['बाहर', 'रात', 'बारे', 'हो']]
Predicte

Source: in have money love alone son this get job
Actual: [['खुश', 'जाते', 'रहा।', 'मुझे', 'किया', 'आता', 'अपना', 'कर', 'दो', 'हो']]
Predicted: []
BLEU Score: 0

Source: i father next very
Actual: [['नहीं', 'ने', 'का', 'कि', 'ना']]
Predicted: []
BLEU Score: 0

Source: meeting staying
Actual: [['कुछ', 'तुम्हें', 'बनाया', 'वह', 'सा']]
Predicted: []
BLEU Score: 0

Source: children i about can as
Actual: [['से', 'मज़े', 'था']]
Predicted: []
BLEU Score: 0

Source: is being the stayed
Actual: [['में', 'गाँव', 'का', 'डाल', 'हम']]
Predicted: []
BLEU Score: 0

Source: i happened you easy and
Actual: [['नहीं', 'कम', 'नौ', 'जन्मदिन', 'हम']]
Predicted: []
BLEU Score: 0

Source: i through the our take
Actual: [['इस', 'चाहता', 'को', 'काम', 'हर', 'कहा', 'उसने']]
Predicted: []
BLEU Score: 0

Source: summer
Actual: [['जानता']]
Predicted: []
BLEU Score: 0

Source: his he doesn t nice languages
Actual: [['था।', 'परीक्षा', 'ज़िम्मेदार', 'मैं', 'होंगे।', 'अपनी', 'क्या', 'पास']]
Predicted: []
BLEU Score: 0


Source: do good first of out year
Actual: [['में', 'किसी', 'पसंद', 'सारा', 'के', 'पता।', 'करनी', 'है।']]
Predicted: []
BLEU Score: 0

Source: i choose to
Actual: [['नहीं', 'काफ़ी', 'छुट्टी', 'पापा', 'बहुत']]
Predicted: []
BLEU Score: 0

Source: at he wants mind can days
Actual: [['उस', 'कर', 'हो।', 'क्या', 'लोग', 'कर', 'हो।', 'है।']]
Predicted: []
BLEU Score: 0

Source: will that talk of right what yesterday you where day
Actual: [['थे।', 'कल', 'लेना', 'है', 'यह', 'की', 'दोनो', 'परसों', 'हो']]
Predicted: []
BLEU Score: 0

Source: this father care to
Actual: [['उसकी', 'माफ़', 'गए', 'समस्या']]
Predicted: []
BLEU Score: 0

Source: at my to from they what
Actual: [['गया।', 'मेज़', 'कोई', 'हैं।', 'मेरा', 'है', 'यह', 'जल्द', 'रहा']]
Predicted: []
BLEU Score: 0

Source: already of favorite live mary away exam please it this cooked talks wish state
Actual: [['खतम', 'हाथ', 'को', 'दूर', 'मुझे', 'कठिन', 'के', 'अचानक', 'साफ़', 'तुमने', 'उसकी', 'फूल', 'मेरे', 'अपने', 'पर', 'ज़्यादा', 'उसके', 'जीत']

Source: is write what a flying
Actual: [['लिए', 'नफ़रत', 'बाद', 'आपके']]
Predicted: []
BLEU Score: 0

Source: do angry t an follow party
Actual: [['में', 'कृपया', 'मैं', 'सीखना', 'दरवाज़ा', 'है।']]
Predicted: []
BLEU Score: 0

Source: in have what empty
Actual: [['टॉम', 'बातचीत', 'है।']]
Predicted: []
BLEU Score: 0

Source: his disappointed legs it home advantage telephone don able
Actual: [['था।', 'तुम्हें', 'चला', 'पहले', 'कुछ', 'मुझे', 'उठते', 'वह', 'मेहनत', 'है।']]
Predicted: []
BLEU Score: 0

Source: i with night of me up
Actual: [['नहीं', 'बात', 'मेरे', 'मुझे', 'देखने', 'हो।', 'बहुत']]
Predicted: []
BLEU Score: 0

Source: i father older the swollen
Actual: [['नहीं', 'खबर', 'की।', 'बहुत']]
Predicted: []
BLEU Score: 0

Source: is spoken used get food
Actual: [['लिए', 'भाई', 'अपना', 'हमे', 'बहनें', 'उसने']]
Predicted: []
BLEU Score: 0

Source: i job she doing take
Actual: [['नहीं', 'काम', 'उसे', 'बता', 'मुझे', 'वैसा']]
Predicted: []
BLEU Score: 0

Source: your our are home t open
Ac

Source: i on you isn we while
Actual: [['नहीं', 'हमे', 'अफ़वाह', 'एक', 'चीनी', 'बहुत']]
Predicted: []
BLEU Score: 0

Source: i are important what sunday s has it return
Actual: [['नहीं', 'तुम', 'दोनों', 'अगर', 'बेटे', 'अपनी', 'वे', 'पछतावा', 'करोगे', 'उसको', 'वे', 'कि', 'चाहिए']]
Predicted: []
BLEU Score: 0

Source: before was to long
Actual: [['हीरा', 'कहाँ', 'दरवाज़ा', 'क्या']]
Predicted: []
BLEU Score: 0

Source: me august he an story offer exhale
Actual: [['हैं', 'बनना', 'पुलिस', 'है', 'इतने', 'पढ़ा', 'हो']]
Predicted: []
BLEU Score: 0

Source: tomorrow and t
Actual: [['आप', 'ऐसा', 'तर']]
Predicted: []
BLEU Score: 0

Source: do died going death all do clothes the known
Actual: [['में', 'होकर', 'भविष्य', 'कि', 'जल्दी', 'वैसे']]
Predicted: []
BLEU Score: 0

Source: road move s is door tall every some as short
Actual: [['उम्मीद', 'तुम्हें', 'आसानी', 'करने', 'में', 'अंडे', 'पत्तियों', 'मैं', 'जाकर', 'किताब', 'इनसान', 'अभी', 'उसने']]
Predicted: []
BLEU Score: 0

Source: taking add he th

Source: your keep he teachers what context
Actual: [['टॉम', 'कितनी', 'दूंगा।', 'है', 'यह', 'संदर्भ', 'कर', 'पता', 'है।']]
Predicted: []
BLEU Score: 0

Source: i are true is rice happy
Actual: [['से', 'घर', 'चावल', 'पर', 'फ़र्क', 'उसने']]
Predicted: []
BLEU Score: 0

Source: do he children later t news every did u
Actual: [['में', 'किताब', 'और', 'निर्णय', 'करो।', 'मैं', 'की', 'इशारा', 'है।']]
Predicted: []
BLEU Score: 0

Source: was to got him you mother a decided
Actual: [['हैं।', 'माफ़', 'गुफ़ा', 'पर', 'अँडे', 'अपने', 'जब', 'क्या']]
Predicted: []
BLEU Score: 0

Source: i watching it who back at is motioned you last
Actual: [['नहीं', 'होमवर्क', 'वह', 'खटखटाया।', 'अपनी', 'वे', 'करने', 'में', 'सुबह', 'हैं।', 'आपको', 'उसने']]
Predicted: []
BLEU Score: 0

Source: will he a cave we the tv
Actual: [['परेशान', 'का', 'झुकी।', 'सीखने', 'खाना', 'है।']]
Predicted: []
BLEU Score: 0

Source: i tom it buy let don your o
Actual: [['पिता', 'टॉम', 'घुसाना', 'डब्बे', 'वह', 'घुसा']]
Predicted: []
BLEU Sc

Source: the week my how promote
Actual: [['जो', 'गर्मी', 'करवाया।', 'दो', 'हो']]
Predicted: []
BLEU Score: 0

Source: i got right
Actual: [['से', 'ताला', 'है', 'मन', 'मैं', 'साल', 'क्या', 'पास']]
Predicted: []
BLEU Score: 0

Source: was to got his industry
Actual: [['हैं।', 'गया।', 'सके।', 'भूल', 'मुझे', 'था।', 'को', 'स्वाद', 'रहा']]
Predicted: []
BLEU Score: 0

Source: i want you someone to don she contains
Actual: [['नहीं', 'हूँ।', 'उसे', 'कड़वा', 'है', 'रहे', 'लड़की', 'अपने', 'आपको', 'बहुत']]
Predicted: []
BLEU Score: 0

Source: do has it well for me ve should
Actual: [['लिए', 'से', 'हैं', 'मत', 'वह', 'सबका']]
Predicted: []
BLEU Score: 0

Source: will have money a medicine of felt your food
Actual: [['बजे', 'अपना', 'की', 'असफल', 'उड़ाता', 'है।']]
Predicted: []
BLEU Score: 0

Source: s are me problem
Actual: [['में', 'गया', 'सका।', 'थी।']]
Predicted: []
BLEU Score: 0

Source: at had accident in little
Actual: [['न', 'आ', 'ड्यूटी', 'क्या']]
Predicted: []
BLEU Score: 0

Source: i like 

In [111]:
# Calculate the average BLEU score
average_bleu_score = sum(bleu_scores) / len(bleu_scores)
print(f'Average BLEU Score: {average_bleu_score}')

Average BLEU Score: 0.0


In [98]:
#The results you've provided indicate that your model's translations are not aligning 
#well with the actual sentences. This is reflected in the extremely low BLEU scores, 
#many of which are zero. There are several potential reasons for this issue:

# 1.Model Performance: The model may not have learned effectively during training. 
#This could be due to insufficient training data, inadequate model complexity, improper 
#hyperparameter settings, or the model not being trained for enough epochs.
# 2.Data Preprocessing: There might be issues with how the data was preprocessed. 
#For example, if the tokenization and detokenization processes are not consistent 
#or if there's a mismatch in how the data was prepared for training versus how it's being prepared for inference.
#3.Translation Function: The translate function might not be working as intended, 
#especially if the model architecture is complex (like an encoder-decoder with attention).
#The model might require a specific format of input or additional processing steps during inference.
#4.Repetitive Predictions: The model seems to be predicting the same words ('मैं', 'मैं') for different inputs, 
#indicating it might have learned a bias towards these words, possibly due to their frequency in the training data.

In [161]:
# well all of this is clearly not working 

In [162]:
# I originally got the file as a .txt but converted it to csv, i though it would make it easier 
# But clearly its not working .

In [163]:
# I will try Now to use the hin.txt file for this problem and see if it works 

In [116]:
text_file = '/Users/shiveshrajsahu/Desktop/Cs767/cs767A1_SAHU_SHIVESH_HW10/hin-eng/hin.txt'

In [117]:
#data processing 

In [118]:
with open(text_file, encoding="utf8") as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    line = line.replace("¡", "").replace("¿", "")
    parts = line.split("\t")
    if len(parts) >= 2:
        eng, ita = parts[0], parts[1]  # Take only the first two parts
        text_pairs.append([eng, ita])
    else:
        # Optionally, handle lines that don't have the expected format
        print(f"Skipping line: {line}")

In [119]:
np.random.seed(42)  
np.random.shuffle(text_pairs)
sentences_en, sentences_ita = zip(*text_pairs)  


In [120]:
#Now to check

In [121]:
#Printing the first 10 lines of our data set

In [122]:
for i in range(10):
    print(sentences_en[i], "=>", sentences_ita[i])

It's getting dark. You'd better go home. => दिन ढल रहा है। तुम्हें घर जाना चाहिए।
That's too small to fit on your head. => वह तुम्हारे सर के लिए बहुत छोटा है।
Sickness prevented him from going out. => वह बीमारी की वजह से बाहर नहीं जा सका।
You should conform to the rules. => तुम्हें नियमों का पालन करना चाहिए।
I told Tom what he should do, but he didn't do it. => मैंने टॉम को बताया उसे क्या करना चाहिए पर उसने वैसा नहीं किया।
I have lots of work to clear up by the weekend. => मुझे इस हफ़्ते बहुत सारा काम कर के खतम करना है।
These dogs are big. => ये कुत्ते बड़े हैं।
The Japanese eat rice at least once a day. => जापानी दिन में कम-से-कम एक बार चावल खाते है।
Who'd want to kill you? => तुम्हें कौन मारना चाहेगा?
The food's not ready yet. => खाना अभी तक तैयार नहीं हुआ है।


In [123]:
vocab_size = 2000
max_length = 70
text_vec_layer_en = tf.keras.layers.TextVectorization(
    vocab_size, output_sequence_length=max_length)
text_vec_layer_ita = tf.keras.layers.TextVectorization(
    vocab_size, output_sequence_length=max_length)
text_vec_layer_en.adapt(sentences_en)
text_vec_layer_ita.adapt([f"startofseq {s} endofseq" for s in sentences_ita])

In [124]:
#now to split the sentence pairs into a training set, a validation set, and a test set.

In [125]:
num_train_samples = int(0.5 * len(sentences_en)) # 50 %
num_test_samples = int(0.15 * len(sentences_en)) # 15 %
num_val_samples = len(sentences_en) - num_train_samples - num_test_samples # 35 %
X_train = tf.constant(sentences_en[:num_train_samples])
X_valid = tf.constant(sentences_en[num_train_samples : num_train_samples + num_val_samples])

X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_ita[:num_train_samples]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_ita[num_train_samples : num_train_samples + num_val_samples]])
X_test_dec = tf.constant([f"startofseq {s}" for s in sentences_ita[num_train_samples + num_val_samples :]])
Y_train = text_vec_layer_ita([f"{s} endofseq" for s in sentences_ita[:num_train_samples]])
Y_valid = text_vec_layer_ita([f"{s} endofseq" for s in sentences_ita[num_train_samples : num_train_samples + num_val_samples]])

X_test =  sentences_en[num_train_samples + num_val_samples :]
Y_test =  sentences_ita[num_train_samples + num_val_samples :]

In [126]:
#Model building

In [127]:
tf.random.set_seed(42)  
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
embed_size = 128
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_ita(decoder_inputs)
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size,
                                                    mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size,
                                                    mask_zero=True)
encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)
tf.random.set_seed(42)  
encoder = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(256, return_sequences=True, return_state=True))

encoder_outputs, *encoder_state = encoder(encoder_embeddings)
encoder_state = [tf.concat(encoder_state[::2], axis=-1),
                 tf.concat(encoder_state[1::2], axis=-1)]  
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)
attention_layer = tf.keras.layers.Attention()
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

In [128]:
# lets try with 2 epohs

In [129]:
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs],
                       outputs=[Y_proba])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])

model.fit((X_train, X_train_dec), Y_train, epochs=2,
          validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/2


W0000 00:00:1700258019.151829       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


47/47 [==============================] - ETA: 0s - loss: 6.4027 - accuracy: 0.1240

W0000 00:00:1700258034.174755       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


47/47 [==============================] - 22s 392ms/step - loss: 6.4027 - accuracy: 0.1240 - val_loss: 5.8585 - val_accuracy: 0.1287
Epoch 2/2
47/47 [==============================] - 17s 373ms/step - loss: 5.6989 - accuracy: 0.1285 - val_loss: 5.7831 - val_accuracy: 0.1321


In [130]:
# now lets try with 20 epohs

In [131]:
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs],
                       outputs=[Y_proba])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])

model.fit((X_train, X_train_dec), Y_train, epochs=20,
          validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/20


W0000 00:00:1700258064.352234       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


47/47 [==============================] - ETA: 0s - loss: 5.5718 - accuracy: 0.1388

W0000 00:00:1700258078.991894       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


47/47 [==============================] - 21s 380ms/step - loss: 5.5718 - accuracy: 0.1388 - val_loss: 5.7227 - val_accuracy: 0.1411
Epoch 2/20
47/47 [==============================] - 17s 353ms/step - loss: 5.4245 - accuracy: 0.1430 - val_loss: 5.6953 - val_accuracy: 0.1445
Epoch 3/20
47/47 [==============================] - 17s 365ms/step - loss: 5.3182 - accuracy: 0.1469 - val_loss: 5.6743 - val_accuracy: 0.1538
Epoch 4/20
47/47 [==============================] - 17s 366ms/step - loss: 5.2008 - accuracy: 0.1591 - val_loss: 5.6097 - val_accuracy: 0.1673
Epoch 5/20
47/47 [==============================] - 17s 353ms/step - loss: 5.0415 - accuracy: 0.1702 - val_loss: 5.5455 - val_accuracy: 0.1720
Epoch 6/20
47/47 [==============================] - 17s 359ms/step - loss: 4.8486 - accuracy: 0.1842 - val_loss: 5.5614 - val_accuracy: 0.1752
Epoch 7/20
47/47 [==============================] - 17s 361ms/step - loss: 4.6635 - accuracy: 0.1969 - val_loss: 5.4359 - val_accuracy: 0.1850
Epoch 8/20

In [133]:
# saving the model 
model.save('/Users/shiveshrajsahu/Desktop/Cs767/cs767A1_SAHU_SHIVESH_HW10/translataion_model.h10_1')

INFO:tensorflow:Assets written to: /Users/shiveshrajsahu/Desktop/Cs767/cs767A1_SAHU_SHIVESH_HW10/translataion_model.h10_1/assets


INFO:tensorflow:Assets written to: /Users/shiveshrajsahu/Desktop/Cs767/cs767A1_SAHU_SHIVESH_HW10/translataion_model.h10_1/assets


In [134]:
from tensorflow.keras.models import load_model
model = load_model('/Users/shiveshrajsahu/Desktop/Cs767/cs767A1_SAHU_SHIVESH_HW10/translataion_model.h10_1')

2023-11-17 17:01:25.220134: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 14 outputs but the _output_shapes attribute specifies shapes for 44 outputs. Output shapes may be inaccurate.
2023-11-17 17:01:25.799493: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 14 outputs but the _output_shapes attribute specifies shapes for 44 outputs. Output shapes may be inaccurate.
2023-11-17 17:01:25.804571: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond' has 5 outputs but the _output_shapes attribute specifies shapes for 44 outputs. Output shapes may be inaccurate.
2023-11-17 17:01:25.869991: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 14 outputs but the _output_shapes attribute specifies shapes for 44 outputs. Output shapes may be inaccurate.
2023-11-17 17:01:25.923525: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 13 outputs but the _ou

In [136]:
from tensorflow.keras.models import load_model
model = load_model('/Users/shiveshrajsahu/Desktop/Cs767/cs767A1_SAHU_SHIVESH_HW10/translataion_model.h10_1')
def translate(sentence_en):
    translation = ""
    for word_idx in range(70):
        X = np.array([sentence_en])  # encoder input
        X_dec = np.array(["startofseq " + translation])  # decoder input
        y_proba = model.predict((X, X_dec))[0, word_idx]  # last token's probas
        predicted_word_id = np.argmax(y_proba)
        predicted_word = text_vec_layer_ita.get_vocabulary()[predicted_word_id]
        if predicted_word == "endofseq":
            break
        translation += " " + predicted_word
    return translation.strip()

2023-11-17 17:01:44.444556: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 14 outputs but the _output_shapes attribute specifies shapes for 44 outputs. Output shapes may be inaccurate.
2023-11-17 17:01:44.997253: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 14 outputs but the _output_shapes attribute specifies shapes for 44 outputs. Output shapes may be inaccurate.
2023-11-17 17:01:45.001651: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond' has 5 outputs but the _output_shapes attribute specifies shapes for 44 outputs. Output shapes may be inaccurate.
2023-11-17 17:01:45.064151: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 14 outputs but the _output_shapes attribute specifies shapes for 44 outputs. Output shapes may be inaccurate.
2023-11-17 17:01:45.115327: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 13 outputs but the _ou

In [137]:
#Testing/example

In [138]:
translate("The sun is beautiful")

1/1 [==============================] - 1s 1s/step


W0000 00:00:1700258515.659725       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


1/1 [==============================] - 0s 29ms/step


'यह [UNK] है।'

In [139]:
translate("The sea is beautiful")

1/1 [==============================] - 0s 30ms/step


'यह [UNK] है।'

In [140]:
def translate_batch():
    random.seed(42)
    sentence_index = [random.randrange(0, len(X_test) - 1) for _ in range(10)]
    eng_sentences = []
    tr_sentences = []
    #orig_sentences = []
    for idx in sentence_index:
        translation = translate(X_test[idx])
        tr_sentences.append(translation)
        eng_sentences.append(X_test[idx])
    return eng_sentences, tr_sentences 
english_sentences, translated_sentences = translate_batch()

1/1 [==============================] - 0s 30ms/step


In [141]:
for eng_sent, tr_sent in zip(english_sentences, translated_sentences):
    print(f"Sentence in english: {eng_sent}")
    print(f"Sentence translated by model: {tr_sent}")
    print("-" * 50)

Sentence in english: I don't think that it will rain tomorrow.
Sentence translated by model: मैं कल कल कल कल कल कल कल नहीं हूँ।
--------------------------------------------------
Sentence in english: The two languages have a lot in common.
Sentence translated by model: जापान में में में में में में में में में में में में में में में में में हैं।
--------------------------------------------------
Sentence in english: I'm too tired to walk.
Sentence translated by model: मैं तुम्हारे पास फ़ोन सकता हूँ।
--------------------------------------------------
Sentence in english: This is the very book that I wanted to read.
Sentence translated by model: यह जाना जाना है मैं मैं जाना है।
--------------------------------------------------
Sentence in english: Tom didn't have any hair.
Sentence translated by model: टॉम पास पास पास नहीं नहीं है।
--------------------------------------------------
Sentence in english: Your dog is very big.
Sentence translated by model: [UNK] बहुत बहुत बहुत बहुत बहुत ब

In [142]:
# well this seems to be working 

In [143]:
#Calculation of the BLUE scores

In [144]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 16.1 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.6 MB/s eta 0:00:00
  Created wheel for lxml: filename=lxml-4.9.3-cp310-cp310-macosx_11_0_arm64.whl size=1492373 sha256=5b4bbf2c9760058b6463d890e61b045bc0b1229c0e3bb1d5c0a1a889e1a68526
  Stored in directory: /Users/shiveshrajsahu/Library/Caches/pip/wheels/38/0b/56/fd5ffdd76481c9220a131ff39258963d8384599f0109b688d0
Successfully built lxml


In [149]:
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 2.4 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.7/347.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 29.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00


In [151]:
X_blue = X_test[1000:1500]

In [152]:
Y_blue = Y_test[1000:1500]

In [153]:

tr_sentences = []
eng_sentences = []
blue_score = []
orig_sentences = []
for idx in range(len(X_blue)):
    translation = translate(X_blue[idx])
    orig = Y_blue[idx]
    orig_sentences.append(orig)
    blue_sc = sentence_bleu([orig.split()], translation.split(), weights = (0.5, 0.5))
    blue_score.append(blue_sc)
    tr_sentences.append(translation)
    eng_sentences.append(X_blue[idx])

In [154]:
for i in range(len(tr_sentences)):
    print(f"Original sentence: {orig_sentences[i]}")
    print(f"Sentence translated by model: {tr_sentences[i]}")
    print(f"Bleu score: {blue_score[i]}")
    print("-" * 50)

In [155]:
print('Average Blue score:', sum(blue_score)/len(blue_score))

ZeroDivisionError: division by zero

In [158]:
# Okay, Lets try again

In [160]:
import sacrebleu

def translate_batch_and_calculate_bleu():
    random.seed(42)
    sentence_index = [random.randrange(0, len(X_test) - 1) for _ in range(10)] # Or a different number of sentences
    eng_sentences = []
    tr_sentences = []
    bleu_scores = []

    for idx in sentence_index:
        translation = translate(X_test[idx])
        tr_sentences.append(translation)
        eng_sentences.append(X_test[idx])

        # Calculate BLEU score
        ref = [Y_test[idx]]
        bleu_score = sacrebleu.corpus_bleu([translation], [ref]).score
        bleu_scores.append(bleu_score)

    return eng_sentences, tr_sentences, bleu_scores

english_sentences, translated_sentences, bleu_scores = translate_batch_and_calculate_bleu()

# Print results
for eng_sent, tr_sent, bleu in zip(english_sentences, translated_sentences, bleu_scores):
    print(f"Sentence in English: {eng_sent}")
    print(f"Sentence translated by model: {tr_sent}")
    print(f"BLEU score: {bleu}")
    print("-" * 50)

# Calculate average BLEU score
average_bleu_score = sum(bleu_scores) / len(bleu_scores)
print(f"Average BLEU score: {average_bleu_score}")


1/1 [==============================] - 1s 1s/step


W0000 00:00:1700259349.312540       1 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" model: "0" frequency: 2400 num_cores: 10 environment { key: "cpu_instruction_set" value: "ARM NEON" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 16384 l2_cache_size: 524288 l3_cache_size: 524288 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


1/1 [==============================] - 0s 29ms/step
Sentence in English: I don't think that it will rain tomorrow.
Sentence translated by model: मैं कल कल कल कल कल कल कल नहीं हूँ।
BLEU score: 4.990049701936832
--------------------------------------------------
Sentence in English: The two languages have a lot in common.
Sentence translated by model: जापान में में में में में में में में में में में में में में में में में हैं।
BLEU score: 2.4074859035470344
--------------------------------------------------
Sentence in English: I'm too tired to walk.
Sentence translated by model: मैं तुम्हारे पास फ़ोन सकता हूँ।
BLEU score: 3.5275023606301383
--------------------------------------------------
Sentence in English: This is the very book that I wanted to read.
Sentence translated by model: यह जाना जाना है मैं मैं जाना है।
BLEU score: 6.413885305524152
--------------------------------------------------
Sentence in English: Tom didn't have any hair.
Sentence translated by model: टॉम पास पास 

In [ ]:
#Average BLEU score: 4.978174524177378